# Laboratório forense TSE 2026 — Fase 5Q / Opção V

## Seções tardias dentro de municípios e zonas eleitorais

**Questão:** a diferença observada na fase T entre a previsão municipal pré-corte e o voto real pós-corte se explica pela composição das zonas, ou aparece inclusive dentro das mesmas zonas?

Usa somente BUs do **primeiro turno de 2026**, dados H/M corrigidos pela opção V, com hashes SHA-256. **Corte por segundo:** 04/10/2026 às **20:04:39 BRT**. O minuto 20:04 pode conter seções dos dois lados do corte, mas a classificação individual preserva os segundos.

**Limitação essencial:** horário BWEB `DT_BU_RECEBIDO` **não equivale** ao horário de publicação EA20. Um desvio entre modelo e votos observados é uma evidência de composição ou previsibilidade, **não uma demonstração automática de alteração eleitoral**. Mesmo dentro de uma zona, seções não formam necessariamente amostra aleatória da população.

## 1. Fontes, SHA e reconciliação

Este notebook abre os resultados da opção **V** do menu BAT. Se os arquivos estiverem ausentes, execute a opção V; não faça downloads manuais.

In [ ]:
from pathlib import Path
import sys, hashlib, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def repo_root():
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent/"start_windows.bat").is_file():
            return parent
    raise FileNotFoundError("Nao foi localizada a raiz do repositorio TSE.")

REPO = repo_root()
sys.path.insert(0, str(REPO))
from urnas import root_path
ROOT = root_path(None)
BASE = ROOT / "analysis" / "periodicidade_bweb_2026"

def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(4*1024*1024),b""):
            h.update(block)
    return h.hexdigest()

def read_verified_csv(meta):
    p = Path(meta["path"])
    if not p.is_file():
        p = BASE/p.name
    if sha256(p)!=meta["sha256"]:
        raise RuntimeError("SHA-256 difere: "+str(p))
    return pd.read_csv(p,sep=";",encoding="utf-8-sig",dtype={"UF":str,"municipio":str,"zona":str})

qpath=BASE/"phase5q_zone_holdout_2026_summary.json"
tpath=BASE/"phase5p_holdout_geographic_curve_2026_summary.json"
q=json.loads(qpath.read_text(encoding="utf-8"))
t=json.loads(tpath.read_text(encoding="utf-8"))
assert q["status"]=="ZONA_PRE_CORTE_TESTADA_NA_CAUDA_BWEB_2026"
assert q["year"]==t["year"]==2026 and q["round"]==t["round"]==1
assert q["valid_post"]==t["data"]["holdout_post_cut_valid"]
minutes=read_verified_csv(q["output_files"]["minutes"])
muns=read_verified_csv(q["output_files"]["municipalities"])
zones=read_verified_csv(q["output_files"]["zones"])
minutes["data_hora"]=pd.to_datetime(minutes["brt_minute"])
assert int(minutes["post_valid"].sum())==q["valid_post"]
assert int(zones["post_valid"].sum())==q["valid_post"]
assert int(muns["post_valid"].sum())==q["valid_post"]
assert int(zones["post_lula"].sum())==q["lula_post"]
print("OK: SHA-256 de 3 CSV V, primeiro turno e totais reconciliados")
print("Zonas com BUs pós-corte:",len(zones),"| Municípios:",len(muns),"| Válidos pós-corte:",q["valid_post"])

## 2. A prova principal: modelo por município versus zona

Todos os modelos **BR pré**, **UF pré**, **município pré** e **zona pré** são ajustados somente com BUs recebidos antes do corte. As taxas municipais são suavizadas por 1.000 votos de referência da UF, e as zonais por 1.000 votos de referência do município. O modelo **ZONA_final_IN_SAMPLE** conhece a cauda e aparece apenas como controle retrospectivo.

A V deve reproduzir as taxas do modelo municipal da T (checagem de implementação independente), antes de comparar a nova taxa por zona.

In [ ]:
labels={"BR_pre":"Brasil pré-corte","UF_pre":"UF pré-corte",
        "MUNICIPIO_pre":"Município pré-corte","ZONA_pre":"Zona pré-corte",
        "ZONA_final_IN_SAMPLE":"Zona FINAL (oráculo circular)"}
out=[]
for key,values in q["forecast_models"].items():
    if key in t["data"]["forecast_models"]:
        ref=t["data"]["forecast_models"][key]
        assert abs(values["lula_final_error_pp"]-ref["lula_final_error_pp"]) < 1e-6
    out.append({"Modelo":labels[key],
                "Previsto Lula (%)":values["pred_lula_pct"],
                "Real Lula (%)":values["actual_lula_pct"],
                "Erro final (pp)":values["lula_final_error_pp"],
                "RMSE por minuto (pp)":values["weighted_minute_rmse_lula_pp"],
                "Minutos avaliados":values["minute_bins_ge_5000_valids"]})
modelos=pd.DataFrame(out)
display(modelos.round(4))
ax=modelos.plot.bar(x="Modelo",y="RMSE por minuto (pp)",figsize=(11,4),legend=False)
ax.set(title="Previsibilidade de lotes BWEB pós-corte (não é curva EA20)",ylabel="RMSE ponderado (pp)",xlabel="")
ax.grid(axis="y",alpha=.2)
plt.xticks(rotation=15,ha="right")
plt.tight_layout()
plt.show()

## 3. Decomposição contábil do erro municipal

A soma das três parcelas abaixo deve reproduzir exatamente o erro em votos do modelo municipal. Grupos: **municípios sem BU anterior**, **zonas sem BU anterior, em municípios já vistos** e **mesmas zonas com BUs anteriores**. Um valor de desvio positivo significa que a votação observada para Lula excedeu a previsão.

In [ ]:
cats=[]
for key, val in q["categories"].items():
    cats.append({"Grupo":key,"Zonas":val["zones"],"Votos pós-corte":val["post_valid"],
                 "% da cauda":100*val["vote_fraction_of_tail"],
                 "Lula observado (%)":val["post_lula_pct"],
                 "Desvio municipal (votos)":val["municipal_gap_votes"],
                 "Desvio zonal (votos)":val["zone_gap_votes"]})
categories=pd.DataFrame(cats).sort_values("Votos pós-corte",ascending=False)
display(categories.round(2))
expected=-q["forecast_models"]["MUNICIPIO_pre"]["lula_final_error_pp"]*q["valid_post"]/100
actual=categories["Desvio municipal (votos)"].sum()
assert abs(expected-actual)<0.01
print("Total: votos de Lula ACIMA da previsão municipal =",round(actual))
ax=categories.set_index("Grupo")["Desvio municipal (votos)"].plot.bar(figsize=(10,4))
ax.axhline(0,linestyle="--")
ax.set(ylabel="Observado menos previsto (votos)",title="Quem responde pelo desvio da previsão municipal?")
plt.xticks(rotation=12,ha="right")
plt.tight_layout()
plt.show()

## 4. Dentro das mesmas zonas: antes e depois

Aplicamos um filtro descritivo **pré-definido**: no mínimo **1.000 votos válidos antes e 1.000 depois** na mesma zona. Assim evitamos comparar taxas extremamente instáveis com amostras minúsculas. Esse corte é de precisão e não transforma os dois grupos em amostras aleatórias.

In [ ]:
pareadas=zones.loc[(zones["pre_valid"]>=1000)&(zones["post_valid"]>=1000)].copy()
print("Zonas pareadas:",len(pareadas),"/",len(zones))
print("Válidos pós-corte nas zonas pareadas:",int(pareadas["post_valid"].sum()))
print("Mudança média ponderada Lula pós - pré (pp):",q["paired_zones_weighted_delta_post_minus_pre_lula_pp"])
fig,ax=plt.subplots(figsize=(8,6))
ax.scatter(pareadas["pre_lula_pct"],pareadas["post_lula_pct"],
           s=8+50*pareadas["post_valid"]/pareadas["post_valid"].max(),alpha=0.45)
ax.plot([0,100],[0,100],linestyle="--")
ax.set(xlabel="Lula (%) pré-corte, mesma zona",ylabel="Lula (%) pós-corte, mesma zona",
       title="Mudança de composição dentro de zonas com dados antes/depois")
ax.set_xlim(0,100)
ax.set_ylim(0,100)
ax.grid(alpha=.2)
plt.show()
display(pareadas["delta_post_minus_pre_pp"].describe().to_frame("Mudança Lula (pp)").round(3))

## 5. Evolução dos resíduos por minuto

O teste é realizado sobre recebimento BWEB; o gráfico não afirma que esses lotes entraram na publicação EA20 no mesmo minuto. O filtro gráfico usa pelo menos 5.000 votos válidos no minuto, assim como o cálculo de RMSE.

In [ ]:
r=minutes.loc[minutes["post_valid"]>=5000]
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(r["data_hora"],r["mun_residual_lula_pp"],label="Observado menos município pré")
ax.plot(r["data_hora"],r["zone_residual_lula_pp"],label="Observado menos zona pré")
ax.axhline(0,linestyle="--")
ax.set(title="Desvios por minuto de recebimento (> = 5.000 válidos)",
       ylabel="Desvio observado - previsto (pp)",xlabel="Data/hora de recebimento")
ax.legend()
ax.grid(alpha=.2)
fig.autofmt_xdate()
plt.show()
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(minutes["cum_post_valid"]/1e6,minutes["cum_actual_lula_pct"],label="Real")
ax.plot(minutes["cum_post_valid"]/1e6,minutes["cum_mun_pre_lula_pct"],label="Município pré")
ax.plot(minutes["cum_post_valid"]/1e6,minutes["cum_zone_pre_lula_pct"],label="Zona pré")
ax.set(title="Curvas acumuladas apenas da cauda BWEB",xlabel="Milhões de válidos pós-corte",
       ylabel="Lula (%) dentro da cauda")
ax.legend()
ax.grid(alpha=.2)
plt.show()

## 6. Territórios que mais contribuem para os resíduos

Ranking por **votos de diferença**, e não apenas pontos percentuais: evita que uma zona minúscula apareça como principal achado. Os códigos são identificadores oficiais do BWEB; compare com tabela de municípios do TSE antes de atribuir nomes.

In [ ]:
display(muns.assign(abs_gap=muns["mun_residual_lula_votes"].abs())
        .sort_values("abs_gap",ascending=False)
        [["UF","municipio","pre_valid","post_valid","post_lula","mun_residual_lula_votes",
          "zone_residual_lula_votes","unseen_zone_valid"]].head(25).round(2))
display(zones.assign(abs_gap=zones["zone_residual_lula_votes"].abs())
        .sort_values("abs_gap",ascending=False)
        [["UF","municipio","zona","pre_valid","post_valid","post_lula",
          "municipio_residual_lula_votes","zone_residual_lula_votes","classification"]]
        .head(30).round(2))

## 7. Concentração por UF e por categoria de zona

Se a zona melhora a previsão nacional, precisamos saber se a melhoria é generalizada ou se depende de poucos estados.

In [ ]:
estados=zones.groupby("UF",as_index=False).agg(
    post_valid=("post_valid","sum"),post_lula=("post_lula","sum"),
    residual_municipio=("municipio_residual_lula_votes","sum"),
    residual_zona=("zone_residual_lula_votes","sum"))
estados["abs_mun"]=estados["residual_municipio"].abs()
display(estados.sort_values("abs_mun",ascending=False)
        [["UF","post_valid","post_lula","residual_municipio","residual_zona"]].round(1))
fig,ax=plt.subplots(figsize=(11,4))
ordered=estados.sort_values("residual_municipio")
ax.bar(ordered["UF"],ordered["residual_municipio"])
ax.axhline(0,linestyle="--")
ax.set(xlabel="UF",ylabel="Observado menos previsto (votos Lula)",
       title="Distribuição por UF do erro municipal pré-corte")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 8. Diagnóstico e limites do achado

**Interpretação falsificável**: se o modelo zonal reduzir substancialmente o erro que restou do municipal, a composição de zonas dentro dos municípios é uma explicação operacional importante. Se não reduzir, o desvio existe inclusive após esse refinamento e a próxima unidade de análise são as **seções dentro da mesma zona**. Em ambos os casos, precisamos separar geografia, horário de recebimento, heterogeneidade e eventual mecanismo de buffering da publicação EA20.

Não confundir:
- diferença entre voto observado e previsão ≠ voto adulterado;
- `ZONA_final_IN_SAMPLE` ≠ previsão independente;
- `DT_BU_RECEBIDO` ≠ instante de publicação nacional;
- visualização cumulativa lisa ≠ lotes individuais lisos.

**Próxima evidência potencial:** teste de emparelhamento por zona e por características das seções, seguida de auditoria do mecanismo de agregação da série EA20. A fase V diagnostica o recebimento; ainda não explica por si só o R²=0,99044 por seções da publicação nacional.

In [ ]:
print("Fase:",q["version"])
print("Corte:",q["cutoff"])
print("Total pre:",q["valid_pre"],"| total pos:",q["valid_post"])
print("Modelo municipal — erro Lula pp:",q["forecast_models"]["MUNICIPIO_pre"]["lula_final_error_pp"])
print("Modelo zonal — erro Lula pp:",q["forecast_models"]["ZONA_pre"]["lula_final_error_pp"])
print("SHA zonas:",q["output_files"]["zones"]["sha256"])
print("SHA minutos:",q["output_files"]["minutes"]["sha256"])
print("Dados eleitorais originais nao foram modificados.")